# distillation_sim MSD Circuit Visualization

This notebook visualizes the d=3 Steane-code MSD physical circuit used in `distillation_sim`.

What it shows:
- the special 5-qubit fake-input prep used for decoder/channel learning
- that fake-input prep followed by Steane encoding on each patch
- the full lossless physical MSD circuit as built by `distillation_sim`

Interpretation note:
If the five injection qubits are entangled before `append_encoding_circuit()`, then applying the Steane encoder independently to each patch preserves that entanglement in the noiseless/isometric picture. That is the structure `distillation_sim` uses for the MSD special-state path.

In [4]:
import sys
from pathlib import Path

from IPython.display import HTML, display

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in REPO_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate the bloqade-lanes repo root.')

DISTILLATION_SIM_CANDIDATES = [
    Path('/Users/jasonhan/Desktop/qmain/personal-workspace/distillation_sim'),
    REPO_ROOT.parent.parent / 'personal-workspace' / 'distillation_sim',
]
for candidate in DISTILLATION_SIM_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'src' / 'distillation_sim').exists():
        DISTILLATION_SIM_ROOT = candidate
        sys.path.insert(0, str(candidate / 'src'))
        break
else:
    raise FileNotFoundError('Could not locate the distillation_sim repo.')

from distillation_sim.circuits.noise_model import msd_updated_noise_model
from distillation_sim.circuits.steane import SteaneCircuit
from distillation_sim.simulation.sampling import CombinedSampler

print('bloqade-lanes:', REPO_ROOT)
print('distillation_sim:', DISTILLATION_SIM_ROOT)

bloqade-lanes: /Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes
distillation_sim: /Users/jasonhan/Desktop/qmain/personal-workspace/distillation_sim


In [5]:
BASIS = 'X'


def make_lossless_steane_factory(num_logical_qubits: int = 5) -> SteaneCircuit:
    noise = msd_updated_noise_model(
        num_logical_qubits=num_logical_qubits,
        patch_size=7,
        global_scalar=0.0,
        loss_scalar=0.0,
        heat_scalar=0.0,
        new_bb1=True,
        new_1q=True,
        new_xy=True,
        new_cz=True,
        new_move=True,
    )
    return SteaneCircuit(noise, num_logical_qubits=num_logical_qubits)


def lossless_stim_circuit(color_code_circuit) -> object:
    return color_code_circuit.noisy_circuit.circuit.get_lossless_circuit()


def show_svg(stim_circuit, *, height: int = 360) -> None:
    svg_text = str(stim_circuit.diagram('timeline-svg'))
    svg_text = svg_text.replace(
        '<svg ',
        f'<svg style="height:{height}px; width:auto; max-width:none; display:block;" ',
        1,
    )
    display(
        HTML(
            '<div style="overflow-x:auto; width:100%; border:1px solid #ddd; padding:8px; background:white;">'
            + svg_text
            + '</div>'
        )
    )


def print_excerpt(stim_circuit, max_lines: int = 80) -> None:
    lines = str(stim_circuit).splitlines()
    print('\n'.join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f'... ({len(lines) - max_lines} more lines)')


def build_stage_circuits(basis: str = 'X') -> dict[str, object]:
    fake_input = make_lossless_steane_factory()
    fake_input.append_naive_layout()
    fake_input.sh_fake_input_circuit(basis)

    fake_input_plus_encoding = make_lossless_steane_factory()
    fake_input_plus_encoding.append_naive_layout()
    fake_input_plus_encoding.sh_fake_input_circuit(basis)
    fake_input_plus_encoding.append_encoding_circuit()

    full_sampler = CombinedSampler.msd_circuit(
        basis,
        inject_error_prob=0.0,
        error_channel='Zrot',
        distance=3,
        add_echoes=True,
        moving_pattern='to_left',
        global_scalar=0.0,
        apply_loss=False,
        new_bb1=True,
        new_1q=True,
        new_xy=True,
        new_cz=True,
        new_move=True,
    )

    return {
        'fake_input': lossless_stim_circuit(fake_input),
        'fake_input_plus_encoding': lossless_stim_circuit(fake_input_plus_encoding),
        'full_msd': full_sampler.physical_circ.noisy_circuit.circuit.get_lossless_circuit(),
    }


stage_circuits = build_stage_circuits(BASIS)
special_input_qubits = [i * 7 + 6 for i in range(5)]
summary = {
    name: {
        'num_qubits': stim_circuit.num_qubits,
        'num_measurements': stim_circuit.num_measurements,
        'num_detectors': stim_circuit.num_detectors,
        'num_observables': stim_circuit.num_observables,
    }
    for name, stim_circuit in stage_circuits.items()
}

print('basis =', BASIS)
print('special input qubits =', special_input_qubits)
summary

basis = X
special input qubits = [6, 13, 20, 27, 34]


{'fake_input': {'num_qubits': 35,
  'num_measurements': 0,
  'num_detectors': 0,
  'num_observables': 0},
 'fake_input_plus_encoding': {'num_qubits': 35,
  'num_measurements': 0,
  'num_detectors': 0,
  'num_observables': 0},
 'full_msd': {'num_qubits': 35,
  'num_measurements': 35,
  'num_detectors': 15,
  'num_observables': 5}}

## Stage 1: special fake-input prep

This is the basis-dependent 5-qubit entangled input state used for the MSD special-state path before Steane encoding.

In [6]:
show_svg(stage_circuits['fake_input'])

## Stage 2: fake input followed by Steane encoding

This shows the same special input after `append_encoding_circuit()` has encoded each patch. Because the input qubits can already be entangled across patches, the encoded logical blocks remain entangled in the noiseless picture.

In [7]:
show_svg(stage_circuits['fake_input_plus_encoding'])

## Stage 3: full lossless MSD physical circuit

This is the complete physical MSD circuit constructed by `CombinedSampler.msd_circuit(...)` in `distillation_sim`, with the noise scalars set to zero so the Clifford structure is easier to inspect.

In [8]:
show_svg(stage_circuits['full_msd'])

In [9]:
print_excerpt(stage_circuits['full_msd'], max_lines=120)

QUBIT_COORDS(0, 0) 0
QUBIT_COORDS(1, 0) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 0) 3
QUBIT_COORDS(4, 0) 4
QUBIT_COORDS(5, 0) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(0, 1) 7
QUBIT_COORDS(1, 1) 8
QUBIT_COORDS(2, 1) 9
QUBIT_COORDS(3, 1) 10
QUBIT_COORDS(4, 1) 11
QUBIT_COORDS(5, 1) 12
QUBIT_COORDS(6, 1) 13
QUBIT_COORDS(0, 2) 14
QUBIT_COORDS(1, 2) 15
QUBIT_COORDS(2, 2) 16
QUBIT_COORDS(3, 2) 17
QUBIT_COORDS(4, 2) 18
QUBIT_COORDS(5, 2) 19
QUBIT_COORDS(6, 2) 20
QUBIT_COORDS(0, 3) 21
QUBIT_COORDS(1, 3) 22
QUBIT_COORDS(2, 3) 23
QUBIT_COORDS(3, 3) 24
QUBIT_COORDS(4, 3) 25
QUBIT_COORDS(5, 3) 26
QUBIT_COORDS(6, 3) 27
QUBIT_COORDS(0, 4) 28
QUBIT_COORDS(1, 4) 29
QUBIT_COORDS(2, 4) 30
QUBIT_COORDS(3, 4) 31
QUBIT_COORDS(4, 4) 32
QUBIT_COORDS(5, 4) 33
QUBIT_COORDS(6, 4) 34
H 27
SQRT_X 6 13 20 27 34
CZ 6 27 13 34
SQRT_X 27
TICK
CZ 6 13 20 27
SQRT_Y_DAG 13 27
TICK
CZ 13 20 27 34
TICK
I 6 13 20 27 34
TICK
Z 6 13 20 27 34
SQRT_Y 6 13 20 27 34
TICK
SQRT_Y_DAG 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21